# Lab 4, Notebook B - Aligning point clouds

*Linear Algebra for Data Science, UCU 2026. Uses Session 5.2 (the SVD, orthogonal Procrustes $Q = UV^\top$, the Kabsch sign fix) and, for B3(d), PCA from Session 5.1.*

A 3-D scanner, a depth camera or a robot's lidar sees an object in its own coordinate system. Before two scans can be compared, merged or used to locate the robot, they must be brought into one frame by a **rigid motion**: a rotation and a translation. When we know which point corresponds to which, the best rotation comes from one $3\times3$ SVD (B1). When the data are nearly flat, that SVD can return a mirror image instead of a rotation, and we must forbid it (B2). When we do not know the correspondences at all, we guess them, align, and guess again: the **iterative closest point** algorithm (B3). Each step of ICP is the problem of B1.

**Convention (as in Lecture 5.2 and the notes).** Points are the **rows** of a matrix. A rigid motion maps the rows of $X$ to the rows of $XQ + \mathbf 1\mathbf t^\top$, with $Q$ orthogonal and $\mathbf t\in\mathbb R^3$. On a single **column** vector $\mathbf x$ the same motion is $\mathbf x\mapsto Q^\top\mathbf x + \mathbf t$. If $X^\top Y = U\Sigma V^\top$ (both clouds centred), the best orthogonal $Q$ is $UV^\top$.

**How to work.** Most tasks ask you to **predict first, in writing, then compute, then explain any difference**. Write the prediction *before* running the cell. A wrong prediction with an honest explanation earns more at the defence than a correct number you cannot explain.

**Rules.**
- Do not call `np.linalg.inv` or `np.linalg.pinv` anywhere.
- Compute every rotation with `np.linalg.svd` on a $3\times3$ matrix. `scipy.linalg.orthogonal_procrustes` and `scipy.spatial.transform.Rotation` are allowed only where a cell says **check**.
- Find nearest neighbours with `scipy.spatial.cKDTree`.
- Do not change the seeds.
- Before you submit, run `Kernel -> Restart & Run All`.

**Marking.** Core B is worth 2 points at the defence; the code point and the extension are shared with Notebook A.

**Contents.** B1 Procrustes with known correspondences | B2 When the best fit is a mirror: Kabsch | B3 Unknown correspondences: ICP | Extension E2 (shape from a moving camera: Tomasi-Kanade). Attempt **at most one** extension in the whole lab.

| | |
|---|---|
| **Team member 1** | *name, surname* |
| **Team member 2** | *name, surname* |
| **Extension attempted** (at most one in the whole lab) | *none / E1 / E2* |
| **Late days used on this lab** | *0* |
| **Runs top to bottom from a clean kernel** | *yes / no* |

In [ ]:
import sys, time
import numpy as np
import scipy, scipy.linalg as sl
from scipy.spatial import cKDTree
import matplotlib, matplotlib.pyplot as plt

print("python", sys.version.split()[0], "| numpy", np.__version__, "| scipy", scipy.__version__,
      "| matplotlib", matplotlib.__version__)
np.set_printoptions(precision=4, suppress=True, linewidth=110)


def make_fish(n, seed):
    # n points on the surface of an asymmetric 'fish': an ellipsoid body, a fin on top, a forked tail bent to one side
    rng = np.random.default_rng(seed)
    nb, nf = int(0.7 * n), int(0.15 * n); nt = n - nb - nf
    z = rng.standard_normal((nb, 3)); z /= np.linalg.norm(z, axis=1, keepdims=True)
    body = z * np.array([2.0, 1.2, 0.5])
    a, b = rng.random(nf), rng.random(nf); m = a + b > 1; a[m], b[m] = 1 - a[m], 1 - b[m]
    P0, P1, P2 = np.array([0.3, 0, 0.5]), np.array([-0.9, 0, 0.45]), np.array([-0.6, 0, 1.1])
    fin = P0 + a[:, None] * (P1 - P0) + b[:, None] * (P2 - P0) + 0.03 * rng.standard_normal((nf, 3))
    s = rng.random(nt); side = np.where(rng.random(nt) < 0.5, 1, -1)
    tail = np.c_[-2.0 - 0.9 * s, 0.25 * s + 0.5 * side * s ** 1.5, 0.45 * side * s] + 0.04 * rng.standard_normal((nt, 3))
    return np.vstack([body, fin, tail])


def rot(axis, deg):
    # rotation by deg degrees about axis, ACTING ON COLUMN VECTORS: x -> R x (Rodrigues' formula)
    k = np.asarray(axis, float); k /= np.linalg.norm(k); th = np.deg2rad(deg)
    K = np.array([[0, -k[2], k[1]], [k[2], 0, -k[0]], [-k[1], k[0], 0]])
    return np.eye(3) + np.sin(th) * K + (1 - np.cos(th)) * K @ K


def angle_deg(Q):
    # rotation angle of a 3x3 rotation matrix, in degrees: trace Q = 1 + 2 cos(angle)
    return np.degrees(np.arccos(np.clip((np.trace(Q) - 1) / 2, -1, 1)))


def rms(A, B):
    # root-mean-square distance between corresponding rows of A and B
    return np.sqrt(np.mean(np.sum((A - B) ** 2, axis=1)))


def show(*clouds, labels=None, title=""):
    fig = plt.figure(figsize=(5.5, 4)); ax = fig.add_subplot(projection="3d")
    for i, C in enumerate(clouds):
        ax.scatter(*C.T, s=3, label=None if labels is None else labels[i])
    ax.set_box_aspect(np.ptp(np.vstack(clouds), axis=0)); ax.set_title(title)
    if labels: ax.legend()
    plt.tight_layout(); plt.show()


X = make_fish(800, seed=1)            # the model: 800 points, rows are points
n = len(X)
print("model:", X.shape, " centred singular values / sqrt(n):", np.linalg.svd(X - X.mean(0), compute_uv=False) / np.sqrt(n))
show(X, title="the model")

---
## B1. Procrustes with known correspondences

A scanner has measured the same 800 points, in the same order, after the fish was moved. The manual says the motion was the rotation $R$ by $40^\circ$ about the axis $(1,2,2)$, acting on column vectors as $\mathbf x\mapsto R\mathbf x$, followed by the shift $\mathbf t = (5,-2,1)$. The measurements carry noise of size $\text{sd} = 0.02$ per coordinate.

**(a)** In our row convention the planted motion is $Y = XQ_{\text{true}} + \mathbf 1\mathbf t^\top + E$. Which matrix is $Q_{\text{true}}$, in terms of $R$? Then implement `procrustes(X, Y)`, which centres both clouds, computes the SVD of the $3\times3$ matrix $\widetilde X^\top\widetilde Y$, and returns $Q = UV^\top$, the translation $\mathbf t$, and the singular values. (Section 2 of Tutorial 5.2 proves that $UV^\top$ is optimal; here you only use it.)

In [ ]:
R_true = rot([1, 2, 2], 40)
t_true = np.array([5.0, -2.0, 1.0])
sd = 0.02

In [ ]:
Q_true = None   # YOUR CODE HERE: the row-convention matrix of the planted motion

def procrustes(X, Y):
    # returns Q (orthogonal), t, s (singular values of the centred X'Y) with  X Q + 1 t' ~ Y
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# check
Y = X @ Q_true + t_true + sd * np.random.default_rng(2).standard_normal(X.shape)
Q, t, s = procrustes(X, Y)
R_sp, _ = sl.orthogonal_procrustes(X - X.mean(0), Y - Y.mean(0))
assert np.allclose(Q, R_sp) and np.allclose(Q.T @ Q, np.eye(3))
print(f"det Q = {np.linalg.det(Q):+.0f};  rotation error {angle_deg(Q @ Q_true.T):.3f} deg;  "
      f"translation error {np.linalg.norm(t - t_true):.4f}")

**(b)** *Predict, then compute.* Lecture 5.2 gives the minimal value
$$\min_Q\|\widetilde XQ - \widetilde Y\|_F^2 = \|\widetilde X\|_F^2 + \|\widetilde Y\|_F^2 - 2(\sigma_1+\sigma_2+\sigma_3).$$
Predict two things before running: whether the residual of your fit will equal this formula exactly or only approximately, and the size of the **mean squared residual per point**, $\frac1n\|XQ + \mathbf 1\mathbf t^\top - Y\|_F^2$, in terms of $\text{sd}$. Then compute both.

**Your prediction** *(write it before running the next cell)*:

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Your answer.** *Write it here.*

**(c)** *Why centre?* Run your SVD **without** centring, on $X^\top Y$ itself, and report the rotation error of $UV^\top$. Explain the failure by writing $X^\top Y$ in terms of $\widetilde X^\top\widetilde Y$ and the two means.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Your answer.** *Write it here.*

**(d)** *Orientation.* Implement `move(P, Q, t)` for a matrix of points in rows. Then a teammate writes the motion for one column vector as `Q @ x + t`. Compute the RMS distance between the moved model and $Y$ with the teammate's version, `X @ Q.T + t`, and with yours. Which motion does the teammate's code apply, and by how many degrees is it off?

In [ ]:
def move(P, Q, t):
    # YOUR CODE HERE
    raise NotImplementedError

# YOUR CODE HERE: the two RMS distances and the angle between the two rotations

**Your answer.** *Write it here.*

**(e)** *Predict, then compute.* How should the rotation error scale with the noise level and with the number of points? Predict a formula of the form $\text{error} \propto \text{sd}^a n^b$. Then measure the mean rotation error over 20 noise draws (seeds `100, ..., 119`) for $n = 10, 100, 1000, 10\,000$ points (`make_fish(n, seed=1)`) and $\text{sd} = 0.01, 0.1$. Fit $a$ and $b$ from the table.

**Your prediction** *(write it before running the next cell)*:

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Your answer.** *Write it here.*

---
## B2. When the best fit is a mirror: Kabsch

In practice the scanner does not see the 800 surface points with known labels. Instead, six **fiducial markers** are glued to the flat metal plate the fish lies on, at known positions $M$ in the model frame. The scanner finds the markers (noise $\text{sd} = 0.02$), Procrustes on the six markers gives the motion, and the motion is then applied to the whole model. The plate is flat up to a machining tolerance $h$: the markers' heights differ from the plane $z = -0.5$ by at most $h$.

In [ ]:
def markers(h):
    xy = np.array([[1.5, 0.8], [1.2, -0.9], [-0.4, 1.0], [-1.0, -0.7], [-2.2, 0.3], [0.2, 0.1]])
    return np.c_[xy, -0.5 + h * np.array([1, -1, 0.5, -0.5, 0.2, -0.2])]

M = markers(0.01)
M_meas = M @ Q_true + t_true + 0.02 * np.random.default_rng(1003).standard_normal(M.shape)
truth = X @ Q_true + t_true          # where the fish really is
print("centred marker singular values:", np.linalg.svd(M - M.mean(0), compute_uv=False))

**(a)** *Predict, then compute.* Run `procrustes(M, M_meas)`. Before running, predict whether $\det Q$ will be $+1$. Report $\det Q$, the RMS marker residual, and the RMS distance between the moved fish `move(X, Q, t)` and `truth`. Show both clouds with `show`. What has happened to the fish?

**Your prediction** *(write it before running the next cell)*:

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Your answer.** *Write it here.*

**(b)** Implement `kabsch(X, Y)`, which returns the best **rotation** ($\det Q = +1$) and translation: if $\det(UV^\top) = -1$, use $Q = U\operatorname{diag}(1, 1, -1)V^\top$ (Lecture 5.2, stated without proof). *Predict before running:* by how much will the **squared** marker residual increase compared with (a)? Use the statement in the lecture and the singular values printed in (a). Then compute the increase, the fish RMS and the rotation error.

**Your prediction** *(write it before running the next cell)*:

In [ ]:
def kabsch(X, Y):
    # YOUR CODE HERE
    raise NotImplementedError

# YOUR CODE HERE: the increase of the squared marker residual, fish RMS, rotation error

**Your answer.** *Write it here.*

**(c)** *Predict, then compute.* How often does Procrustes return a reflection? For $h = 0.01, 0.03, 0.1, 0.3$, repeat the measurement with seeds `1000, ..., 1999` and report the fraction of reflections, together with the smallest singular value $\sigma_3(\widetilde M)$ of the centred marker matrix. Before running, predict how the fraction depends on $h$, and at roughly which $\sigma_3(\widetilde M)$ it should become negligible, given $\text{sd} = 0.02$.

**Your prediction** *(write it before running the next cell)*:

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Your answer.** *Write it here.*

**(d)** The reflection fits the six markers *better* than the true rotation does, yet moves the fish by more than a metre. Explain both facts. What would you change in the marker layout, and why does Kabsch alone not solve the underlying problem?

**Your answer.** *Write it here.*

---
## B3. Unknown correspondences: iterative closest point

Now the scanner returns a fresh scan of the fish: 800 **different** surface points (another sample of the same surface), in random order, with noise $0.01$. Nothing says which scan point belongs to which model point. ICP alternates two steps that both lower the same objective $\sum_i\min_j\|Q^\top\mathbf x_i + \mathbf t - \mathbf y_j\|^2$:

1. **match**: for each moved model point, find the nearest scan point;
2. **align**: with these matches fixed, find the best rotation and translation (B2's `kabsch`).

It stops when the RMS distance changes by less than `tol`.

In [ ]:
def scan(Q, t, n=800, seed=2, sd=0.01, keep=None):
    # a new sample of the fish surface, moved by (Q, t), noisy, shuffled; keep: optional mask on the sample
    P = make_fish(n, seed)
    if keep is not None: P = P[keep(P)]
    r = np.random.default_rng(seed + 50)
    Ys = P @ Q + t + sd * r.standard_normal(P.shape)
    return Ys[r.permutation(len(Ys))]

**(a)** Implement `icp(X, Y, Q, t, iters=60, tol=1e-7, trim=1.0)`, starting from the motion $(Q, \mathbf t)$. It returns the final $Q$, $\mathbf t$ and the history of RMS nearest-neighbour distances (one value per iteration, plus the final one). Build one `cKDTree` on $Y$ outside the loop. With `trim < 1`, the align step uses only the fraction `trim` of matches with the smallest distances (and the RMS is computed over the same matches); you will need this in (e).

In [ ]:
def icp(X, Y, Q, t, iters=60, tol=1e-7, trim=1.0):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# check: a 30-degree motion, started from the identity and the difference of the means
Q30 = rot([1, 2, 2], 30).T
Ys = scan(Q30, t_true)
Qi, ti, hist = icp(X, Ys, np.eye(3), Ys.mean(0) - X.mean(0))
print(f"{len(hist) - 1} iterations; RMS {hist[0]:.4f} -> {hist[1]:.4f} -> {hist[2]:.4f} -> ... -> {hist[-1]:.4f}")
print(f"rotation error {angle_deg(Qi @ Q30.T):.2f} deg;  RMS never increased: {np.all(np.diff(hist) <= 1e-12)}")
plt.figure(figsize=(5, 3)); plt.semilogy(hist, "o-", ms=3); plt.xlabel("iteration"); plt.ylabel("RMS nearest-neighbour distance")
plt.tight_layout(); plt.show()

**(b)** Two questions about the run above. Why can the RMS never increase from one iteration to the next? And the scan noise is $0.01$: why does the final RMS stop near $0.09$ rather than near $0.01$, although the rotation is recovered to about half a degree?

**Your answer.** *Write it here.*

**(c)** *Predict, then compute.* The **basin of attraction** of ICP is the set of starting errors from which it finds the right answer. Before running, predict the largest rotation angle from which ICP, started at the identity, still succeeds reliably. Then, for angles $0^\circ, 15^\circ, \dots, 180^\circ$ and the 20 random axes below, run ICP from the identity and the difference of the means, and report the fraction of successes (final rotation error below $2^\circ$) and the median final RMS of the failures. Could you detect a failure without knowing the truth?

**Your prediction** *(write it before running the next cell)*:

In [ ]:
axes = np.random.default_rng(7).standard_normal((20, 3))

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Your answer.** *Write it here.*

**(d)** *A better start, from Module 5.* The principal directions of the centred model (the right singular vectors $V_X$) and of the centred scan ($V_Y$) are the axes of the fish in the two frames, each determined only up to sign. Implement `pca_candidates(X, Y)`, which returns the four **rotations** $Q = V_X D V_Y^\top$ with $D$ a diagonal matrix of signs. (Why four and not eight?) Then implement `icp_pca(X, Y)`, which runs ICP from each candidate, with $\mathbf t = \bar{\mathbf y} - Q^\top\bar{\mathbf x}$ in row form, and keeps the run with the smallest final RMS. Repeat the sweep of (c) for angles $0^\circ, 30^\circ, \dots, 180^\circ$.

In [ ]:
def pca_candidates(X, Y):
    # YOUR CODE HERE
    raise NotImplementedError

def icp_pca(X, Y):
    # YOUR CODE HERE
    raise NotImplementedError

# YOUR CODE HERE: the sweep

**Your answer.** *Write it here.*

**(e)** *Predict, then compute.* A real scan sees only part of the object, and picks up stray points. The scan below sees only the side of the fish with $y > -0.3$ (in the model frame), and 10% extra points scattered uniformly around it. Start ICP from the identity and the difference of the **medians** (why not the means?), with up to 300 iterations, and sweep `trim` over $1.0, 0.9, \dots, 0.3$. Report the rotation error and the fish RMS against `truth30 = move(X, Q30, t_true)`. Before running, predict which values of `trim` will work, using the fraction of the fish the scanner sees (printed below).

In [ ]:
visible = lambda P: P[:, 1] > -0.3
Yv = scan(Q30, t_true, keep=visible)
r = np.random.default_rng(52)
lo, hi = Yv.min(0) - 0.5, Yv.max(0) + 0.5
Yp = np.vstack([Yv, lo + (hi - lo) * r.random((len(Yv) // 10, 3))])
Yp = Yp[r.permutation(len(Yp))]
truth30 = move(X, Q30, t_true)
print(f"scan: {len(Yv)} surface points + {len(Yp) - len(Yv)} stray points; "
      f"the scanner sees {visible(make_fish(800, 2)).mean():.1%} of the fish")

**Your prediction** *(write it before running the next cell)*:

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Your answer.** *Write it here.*

**(f)** *Predict, then measure.* One ICP iteration does three things: query $n$ points in the tree, form the $3\times3$ matrix $\widetilde X^\top\widetilde Y$, and take its SVD; building the tree is done once. Predict how each cost grows with $n$ (model and scan both of size $n$), and which dominates. Then time them, on clouds that are already roughly aligned, for $n = 10^3, 10^4, 10^5$. Finally estimate the time of brute-force matching (all $n^2$ distances) for $n = 10^4$ from a $2000\times10^4$ block.

**Your prediction** *(write it before running the next cell)*:

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Your answer.** *Write it here.*

---
# Extension - attempt **at most one** in the whole lab

End your extension with a paragraph headed **Finding**: one claim you can defend at the defence, supported by a number in this notebook.

## E2. Shape from a moving camera: Tomasi-Kanade

*This topic was not lectured; the introduction below is all you need.*

A camera moves around a rigid object and tracks $n$ points through $F$ frames. Far from the object the camera is close to **orthographic**: in frame $f$ it sees the point $\mathbf p$ (a column vector in the object frame) at the image coordinates $(\mathbf i_f^\top\mathbf p + a_f,\ \mathbf j_f^\top\mathbf p + b_f)$, where $\mathbf i_f, \mathbf j_f$ are the first two rows of the camera's rotation (two orthonormal vectors) and $(a_f, b_f)$ is a shift. Stack the $2F$ image rows into the **measurement matrix** $W\in\mathbb R^{2F\times n}$ (column $p$ holds the track of point $p$). After subtracting each row's mean,
$$\widetilde W = \underbrace{\begin{pmatrix}\mathbf i_1^\top\\ \mathbf j_1^\top\\ \vdots\\ \mathbf j_F^\top\end{pmatrix}}_{M\ (2F\times3)}\ \underbrace{\begin{pmatrix}\widetilde{\mathbf p}_1 & \cdots & \widetilde{\mathbf p}_n\end{pmatrix}}_{S\ (3\times n)} + \text{noise},$$
so $\widetilde W$ has rank 3 in the absence of noise (Tomasi and Kanade, 1992). The camera motion $M$ and the shape $S$ are both unknown; the SVD recovers them.

In [ ]:
P_true = make_fish(300, seed=3)            # the object (unknown to the method)
F = 15
cams = [rot([1, 0, 0], 20 * np.sin(2 * np.pi * f / F)) @ rot([0, 0, 1], -40 + 80 * f / (F - 1)) for f in range(F)]
shifts = np.random.default_rng(8).uniform(-1, 1, (2 * F, 1))

def measurements(sd):
    W = np.vstack([P_true @ C[:2].T for C in cams]).reshape(F, -1, 2).transpose(0, 2, 1).reshape(2 * F, -1)
    return W + shifts + sd * np.random.default_rng(9).standard_normal((2 * F, len(P_true)))

W = measurements(0.003)
print("W:", W.shape)

**(a)** *Predict, then compute.* Centre the rows of $W$ and compute its singular values. Predict how many stand out, and the level of the others, using the noise floor $\text{sd}(\sqrt{m} + \sqrt{n})$ of Notebook A (A2). Compare.

**Your prediction** *(write it before running the next cell)*:

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Your answer.** *Write it here.*

**(b)** Set $\widehat M = U_3\Sigma_3^{1/2}$ and $\widehat S = \Sigma_3^{1/2}V_3^\top$, so that $\widehat M\widehat S$ is the best rank-3 approximation of $\widetilde W$. Show that $\widehat MA$ and $A^{-1}\widehat S$ fit the data equally well for **every** invertible $3\times3$ matrix $A$. Check numerically that the rows of $\widehat M$ are not pairs of orthonormal vectors, so $\widehat S$ is the shape only up to an unknown linear distortion.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Your answer.** *Write it here.*

**(c)** *The metric upgrade.* We want $A$ such that the rows of $M = \widehat MA$ satisfy $\mathbf i_f^\top\mathbf i_f = \mathbf j_f^\top\mathbf j_f = 1$ and $\mathbf i_f^\top\mathbf j_f = 0$. With $L := AA^\top$ these conditions are **linear** in the six entries of the symmetric matrix $L$: for instance $\widehat{\mathbf m}_{2f-1}^\top L\,\widehat{\mathbf m}_{2f-1} = 1$. Set up the $3F\times6$ system, solve it by least squares (`np.linalg.lstsq`), check that $L$ is positive definite, and recover $A$ as its Cholesky factor (Module 4). Then compute $S = A^{-1}\widehat S$ with a triangular solve and check the orthonormality conditions for $M$.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Your answer.** *Write it here.*

**(d)** Compare the recovered shape (the rows of $S^\top$) with the true centred object, after aligning them with `procrustes` and with `kabsch` from B1-B2. Which of the two must you use here, and why? (*Hint:* is the mirror image of the object, seen by the mirrored camera, consistent with the same images?) Then mirror the reconstruction, $M \to MD$, $S \to DS$ with $D = \operatorname{diag}(1,1,-1)$: check that it fits the images exactly as well and still satisfies the orthonormality conditions, and align it to the truth in both ways. Repeat for $\text{sd} = 0.03$ and report the RMS shape error relative to the size of the object.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Your answer.** *Write it here.*

---
**Before you submit:** run `Kernel -> Restart & Run All`, check that every written answer is filled in, and make sure your extension (if any) ends with a **Finding**.